# Misplacement scan — fields an institution maps the wrong content into

Whether a whole column of an institution's export sits in the wrong SPECTRUM field, as opposed to a value mentioned inside prose. Two detectors in `mds_norm.pipeline.misplacement_scan`: **typed** (probe spans covering at least `CELL_COVERAGE` of the cell) and **distributional** (the field the rest of the corpus keeps a short value in, with support counted in institutions and the judged institution left out). Both are scored by lift over the leave-one-out rate, a one-sided binomial test with BH adjustment, and whether the destination field is empty on the affected records.

In [1]:
import polars as pl

from mds_norm.paths import FIELD_STATS, MISPLACEMENT_EXAMPLES, MISPLACEMENT_PAIRS, PROBE_CANDIDATES
from mds_norm.pipeline import misplacement_scan as ms

pl.Config(tbl_rows=30, fmt_str_lengths=60, tbl_width_chars=240, float_precision=4)
field_stats = pl.scan_parquet(FIELD_STATS)

## 1. Typed hits

Span lengths summed per cell; cells above `CELL_COVERAGE` are the misfiled population.

In [2]:
cells = (
    pl.scan_parquet(PROBE_CANDIDATES)
    .join(field_stats.select("node_id", "char_count"), on="node_id")
    .group_by("node_id", "field_type", "group")
    .agg(cov=pl.col("candidate").str.len_chars().sum() / pl.col("char_count").first())
    .collect(engine="streaming")
)
cells.select(
    spans=pl.len(),
    prose=(pl.col("cov") < 0.2).mean(),
    partial=((pl.col("cov") >= 0.2) & (pl.col("cov") < ms.CELL_COVERAGE)).mean(),
    whole_cell=(pl.col("cov") >= ms.CELL_COVERAGE).mean(),
)

spans,prose,partial,whole_cell
u32,f64,f64,f64
685606,0.9387,0.0569,0.0044


In [3]:
typed = ms.typed_hits(field_stats, pl.scan_parquet(PROBE_CANDIDATES)).collect(engine="streaming")
(
    typed.group_by("data_source", "field_type", "home_field")
    .agg(hits=pl.len(), example=pl.col("value").first())
    .sort("hits", descending=True)
    .head(15)
)

data_source,field_type,home_field,hits,example
enum,str,str,u32,str
"""University of Aberdeen Collections""","""spectrum/technical_attribute""","""spectrum/dimension""",1191,"""diameter 40 mm; height 170 mm"""
"""Victoria and Albert Museum""","""spectrum/brief_description""","""spectrum/object_production_date""",533,"""circa 1420"""
"""Amgueddfa Cymru - Museum Wales""","""spectrum/object_production_note""","""spectrum/object_production_date""",205,"""21 January 1904"""
"""Culture Coventry""","""spectrum/object_production_note""","""spectrum/object_production_date""",167,"""1 Aug 1964"""
"""Victoria and Albert Museum""","""spectrum/object_production_note""","""spectrum/object_production_date""",117,"""dated January 1882"""
"""Wiltshire Museum""","""spectrum/content_note""","""spectrum/object_production_date""",102,"""post 1932"""
"""Amgueddfa Cymru - Museum Wales""","""spectrum/entry_note""","""spectrum/object_production_date""",82,"""Donated November 2000"""
"""Aberdeen Archives, Gallery and Museums""","""spectrum/brief_description""","""spectrum/technical_attribute_measurement""",65,"""18 pages.."""
"""Amgueddfa Cymru - Museum Wales""","""spectrum/field_collection_note""","""spectrum/field_collection_date""",48,"""Excavated October 1973 - March 1974"""


## 2. Distributional hits

`consensus_home` keeps a value only where at least `MIN_HOME_INSTITUTIONS` other institutions agree on one field holding at least `HOME_DOMINANCE` of the support. Values that float across fields never acquire a home.

In [4]:
usage = ms.field_usage(field_stats)
consensus = ms.consensus_home(usage)
print(f"{len(usage):,} (institution, field, value) cells → {len(consensus):,} values homed elsewhere")
consensus.sort("home_institutions", descending=True).head(10)

6,370,932 (institution, field, value) cells → 2,714 values homed elsewhere


data_source,norm,used_field,home_field,home_institutions,total_institutions
enum,str,str,str,u32,u32
"""Wiltshire Museum""","""1775""","""spectrum/associated_date""","""spectrum/object_production_date""",29,36
"""Victoria Art Gallery""","""1775""","""spectrum/associated_date""","""spectrum/object_production_date""",29,36
"""Aberdeen Archives, Gallery and Museums""","""1775""","""spectrum/associated_date""","""spectrum/object_production_date""",29,36
"""Wiltshire Museum""","""1775""","""spectrum/content_date""","""spectrum/object_production_date""",29,36
"""University of Aberdeen Collections""","""1775""","""spectrum/associated_date""","""spectrum/object_production_date""",29,36
"""Amgueddfa Cymru - Museum Wales""","""1775""","""spectrum/ownership_dates""","""spectrum/object_production_date""",29,36
"""Pewsey Heritage Centre""","""1775""","""spectrum/content_date""","""spectrum/object_production_date""",29,36
"""Norfolk Museums Service""","""1775""","""spectrum/content_date""","""spectrum/object_production_date""",29,36
"""Ashmolean Museum""","""weight""","""spectrum/title""","""spectrum/dimension""",27,29


In [5]:
distributional = ms.distributional_hits(field_stats, consensus).collect(engine="streaming")
print(f"{len(distributional):,} occurrences")
distributional.head(5)

109,680 occurrences


data_source,field_type,home_field,record_id,value,detector
enum,str,str,str,str,str
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""3117648a-6c15-3010-9965-dab24b0bddc4""","""early 19th Century""","""distributional"""
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""7a9ac9b8-5d27-34b3-89f4-998460bc4c04""","""early 19th Century""","""distributional"""
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""217e64b2-1f1a-36f7-9558-58c8c5493070""","""early 19th Century""","""distributional"""
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""9d5a3d2b-25b0-33ac-9c9d-9883a7860419""","""early 19th Century""","""distributional"""
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""27edfdad-2c10-39d7-894c-07ceb47f06ab""","""early 19th Century""","""distributional"""


## 3. Scoring

`run()` does both detectors, joins home-field presence per record and writes the three artefacts.

In [6]:
pairs = ms.run(["typed", "distributional"])
pairs.select("data_source", "field_type", "home_field", "detector", "hits", "values", "share", "lift", "q", "peers", "home_absent", "home_occ", "mapping_issue")

[16:26:16] typed: 3,037 whole-cell hits over 14 fields
[16:26:17] distributional: 6,370,932 (institution, field, value) cells
[16:26:18] distributional: 2,714 values the corpus homes elsewhere
[16:26:19] distributional: 109,680 hits
[16:26:20] 40 pairs above the floors, 14 flagged → /home/liam/Documents/university/mds-norm/data/analysis_output/misplacement/field_pairs.parquet
[16:26:20] 13 institutions, 47,122 occurrences


data_source,field_type,home_field,detector,hits,values,share,lift,q,peers,home_absent,home_occ,mapping_issue
enum,str,str,str,u32,u32,f64,f64,f64,u32,f64,u32,bool
"""Dorset Museum & Art Gallery""","""spectrum/responsible_department_section""","""spectrum/content_concept""","""distributional""",30238,1,0.1619,22708.0248,0.0000,1,0.9999,8585,true
"""Royal Armouries""","""spectrum/technical_attribute_measurement""","""spectrum/dimension""","""distributional""",27189,1,0.7165,70651.5616,0.0000,0,0.1038,207156,false
"""Aberdeen Archives, Gallery and Museums""","""spectrum/associated_date""","""spectrum/object_production_date""","""distributional""",11172,506,0.0857,2.0849,0.0000,9,1.0000,0,false
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""distributional""",6676,13,0.2055,1090.2975,0.0000,0,1.0000,0,true
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""","""distributional""",5819,84,0.0723,1.3573,0.0000,9,0.0423,149597,false
"""Culture Perth & Kinross""","""spectrum/content_other_type""","""spectrum/content_concept""","""distributional""",3504,32,0.0478,2443.0507,0.0000,0,0.5722,40636,true
"""Amgueddfa Cymru - Museum Wales""","""spectrum/ownership_dates""","""spectrum/object_production_date""","""distributional""",3484,119,0.0420,185.4863,0.0000,0,0.6926,518951,true
"""University of Aberdeen Collections""","""spectrum/associated_date""","""spectrum/object_production_date""","""distributional""",2400,192,0.0374,0.6030,1.0000,9,1.0000,0,false
"""Amgueddfa Cymru - Museum Wales""","""spectrum/field_collection_date""","""spectrum/object_production_date""","""distributional""",2054,8,0.0087,3.6258,0.0000,3,0.0078,518951,false


## 4. What is flagged

`mapping_issue` is lift ≥ 5, BH q ≤ 0.01 and the home field missing from at least half the affected records. `home_occ` of zero is the strongest case.

In [7]:
flagged = pairs.filter("mapping_issue").sort("hits", descending=True)
flagged.select("data_source", "field_type", "home_field", "detector", "hits", "values", "records", "share", "lift", "peers", "home_absent", "home_occ")

data_source,field_type,home_field,detector,hits,values,records,share,lift,peers,home_absent,home_occ
enum,str,str,str,u32,u32,u32,f64,f64,u32,f64,u32
"""Dorset Museum & Art Gallery""","""spectrum/responsible_department_section""","""spectrum/content_concept""","""distributional""",30238,1,30238,0.1619,22708.0248,1,0.9999,8585
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""distributional""",6676,13,5537,0.2055,1090.2975,0,1.0000,0
"""Culture Perth & Kinross""","""spectrum/content_other_type""","""spectrum/content_concept""","""distributional""",3504,32,3454,0.0478,2443.0507,0,0.5722,40636
"""Amgueddfa Cymru - Museum Wales""","""spectrum/ownership_dates""","""spectrum/object_production_date""","""distributional""",3484,119,3431,0.0420,185.4863,0,0.6926,518951
"""University of Aberdeen Collections""","""spectrum/technical_attribute""","""spectrum/dimension""","""typed""",1191,1070,1191,0.0121,209.9719,0,1.0000,0
"""Thackray Museum of Medicine""","""spectrum/object_production_organisation""","""spectrum/object_production_person""","""distributional""",874,11,864,0.0253,31.7956,2,1.0000,107
"""Chippenham Museum""","""spectrum/associated_person""","""spectrum/object_production_person""","""distributional""",322,7,322,0.0214,6.6842,5,0.5590,6884
"""Wotton-under-Edge Heritage Centre""","""spectrum/owner""","""spectrum/acquisition_method""","""distributional""",270,1,270,0.0208,1090.6086,0,1.0000,0
"""Horniman Museum and Gardens""","""spectrum/field_collection_date""","""spectrum/object_production_date""","""distributional""",215,12,109,0.0283,5.8204,3,0.7860,68437


In [8]:
examples = pl.read_parquet(MISPLACEMENT_EXAMPLES)
examples.join(flagged.select(ms.PAIR_KEYS), on=ms.PAIR_KEYS).select("data_source", "field_type", "home_field", "value", "n").head(30)

data_source,field_type,home_field,value,n
enum,str,str,str,u32
"""Dorset Museum & Art Gallery""","""spectrum/responsible_department_section""","""spectrum/content_concept""","""Library""",30238
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""local history""",4730
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""buildings""",826
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""recreation""",738
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""road""",214
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""rail""",113
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""canal""",31
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""sea""",6
"""Pewsey Heritage Centre""","""spectrum/associated_activity""","""spectrum/content_concept""","""sheep""",5


## 5. Reading the near misses

High lift with a populated home field is a deliberate role distinction. Many `peers` means a corpus-wide convention rather than one institution's fault.

In [9]:
(
    pairs.filter(~pl.col("mapping_issue"))
    .sort("hits", descending=True)
    .select("data_source", "field_type", "home_field", "hits", "share", "lift", "q", "peers", "home_absent", "home_occ")
    .head(20)
)

data_source,field_type,home_field,hits,share,lift,q,peers,home_absent,home_occ
enum,str,str,u32,f64,f64,f64,u32,f64,u32
"""Royal Armouries""","""spectrum/technical_attribute_measurement""","""spectrum/dimension""",27189,0.7165,70651.5616,0.0000,0,0.1038,207156
"""Aberdeen Archives, Gallery and Museums""","""spectrum/associated_date""","""spectrum/object_production_date""",11172,0.0857,2.0849,0.0000,9,1.0000,0
"""Dorset Museum & Art Gallery""","""spectrum/associated_date""","""spectrum/object_production_date""",5819,0.0723,1.3573,0.0000,9,0.0423,149597
"""University of Aberdeen Collections""","""spectrum/associated_date""","""spectrum/object_production_date""",2400,0.0374,0.6030,1.0000,9,1.0000,0
"""Amgueddfa Cymru - Museum Wales""","""spectrum/field_collection_date""","""spectrum/object_production_date""",2054,0.0087,3.6258,0.0000,3,0.0078,518951
"""Norfolk Museums Service""","""spectrum/content_date""","""spectrum/object_production_date""",1644,0.0456,1.9061,0.0000,5,0.7001,172712
"""Fitzwilliam Museum""","""spectrum/acquisition_source""","""spectrum/object_production_person""",1275,0.0056,41.4607,0.0000,0,0.0024,334938
"""Victoria and Albert Museum""","""spectrum/style""","""spectrum/object_production_person""",1015,0.0091,1779.4019,0.0000,0,0.0000,787447
"""Jersey Heritage""","""spectrum/associated_person""","""spectrum/object_production_person""",610,0.0432,15.7731,0.0000,5,0.0016,24987


In [10]:
(
    pairs.group_by("field_type", "home_field")
    .agg(institutions=pl.col("data_source").n_unique(), hits=pl.col("hits").sum(), flagged=pl.col("mapping_issue").sum())
    .sort("hits", descending=True)
    .head(15)
)

field_type,home_field,institutions,hits,flagged
str,str,u32,u32,u32
"""spectrum/responsible_department_section""","""spectrum/content_concept""",2,30269,2
"""spectrum/technical_attribute_measurement""","""spectrum/dimension""",1,27189,0
"""spectrum/associated_date""","""spectrum/object_production_date""",6,20227,0
"""spectrum/associated_activity""","""spectrum/content_concept""",1,6676,1
"""spectrum/content_other_type""","""spectrum/content_concept""",1,3504,1
"""spectrum/ownership_dates""","""spectrum/object_production_date""",1,3484,1
"""spectrum/field_collection_date""","""spectrum/object_production_date""",4,2524,1
"""spectrum/content_date""","""spectrum/object_production_date""",3,1989,0
"""spectrum/acquisition_source""","""spectrum/object_production_person""",1,1275,0


## 6. Where the output goes

`field_pairs.parquet` is the ranked evidence, `hits.parquet` the occurrences, `examples.parquet` the values to read first. Nothing here writes a sidecar; a confirmed pair belongs in the `DEST` map in `mds_norm.utils.patches`.